In [1]:
import networkx as nx

## 1. Pairwise connectivity function

In [2]:
cd ..

c:\Users\tuguldur\Development\Research\ECNDP\Extended-Critical-Node-Detection-Problem


In [3]:
from algorithms.compute_PC import compute_pc

# 2. Create dummy graphs

In [4]:
def create_custom_graph_with_2_comps(terminal_nodes: list[int]) -> nx.Graph:

  G = nx.Graph()

  C1 = [(1, 2), (2, 3), (3, 4)]
  C2 = [(5, 6), (6, 7)]

  G.add_edges_from(C1)
  G.add_edges_from(C2)

  # Add terminal attribute to nodes
  for v in G.nodes:
    if v in terminal_nodes:
      G.nodes[v]["terminal"] = True
    else:
      G.nodes[v]["terminal"] = False
  
  return G

In [5]:
def create_custom_graph_extreme(terminal_nodes, edge_arrangement):
  
  G = nx.Graph()

  if edge_arrangement == 1:
    C = [(1, 2), (1, 4), (2, 5), (2, 3), (3, 6), (5, 6), (4, 5)]
  if edge_arrangement == 2:
    C = [(1, 2), (1, 3), (2, 5), (2, 4), (3, 5), (5, 6), (4, 6)]
  if edge_arrangement == 3:
    C = [(1, 2), (1, 3), (2, 5), (2, 4), (3, 4), (4, 6), (5, 6)]

  G.add_edges_from(C)

  # Add terminal attribute to nodes
  for v in G.nodes:
    if v in terminal_nodes:
      G.nodes[v]["terminal"] = True
    else:
      G.nodes[v]["terminal"] = False
  
  return G

# 3. Greedy algorithm

## 3.1 Greedy algorithm - maximum difference gain

In [6]:
def greedy_algorithm(
    G: nx.Graph, terminals: list[int], 
    k: int, case: 1 | 2) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage

  T : set = set(terminals)
  V : set = set(G.nodes)

  S : set = set()
  pc_deltas : list[int] = []

  for _ in range(k):

    best_node = None
    best_deg = -1
    best_pc = float("inf")

    # print(f"\nK: iteration\n")
    for node in V - S:
      
      pc_Sj = None

      if case == 1:
        # case 1: S in V
        S_j = S | {node}
        pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)

        # print(f"node: {node} and pc Sj : {pc_Sj}")

      if case == 2:
        # case 2: S in V \ T
        if node not in T:
          S_j = S | {node}
          pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)

      # tie-breaker - if two nodes give the same best_pc
      # get the one with higher degree
      # deg = G.degree(node)

      # argmin
      if pc_Sj is not None and pc_Sj < best_pc:
        best_pc = pc_Sj
        best_node = node
        # best_deg = deg
        # print(f"best node: {best_node} and best gain: {best_pc}")

    if best_node is None:
      # No feasible solution
      break 
    
    S.add(best_node)
    pc_deltas.append(best_pc)

  return S, pc_deltas

In [7]:
T = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(T)
G.nodes(data="terminal")

NodeDataView({1: True, 2: True, 3: False, 4: True, 5: False, 6: True, 7: True}, data='terminal')

In [8]:
result = greedy_algorithm(G, T, k=2, case=1)
result

({2, 6}, [1, 0])

In [9]:
result = greedy_algorithm(G, T, k=2, case=2)
result

({3, 5}, [2, 2])

In [10]:
for i in range(3):
  T = [1, 2, 6]
  G = create_custom_graph_extreme(T, edge_arrangement=i + 1)
  G.nodes(data="terminal")

  result = greedy_algorithm(G, T, k=2, case=2)
  print(f"iter: {i + 1} and result: {result}")

iter: 1 and result: ({3, 5}, [3, 1])
iter: 2 and result: ({3, 4}, [3, 3])
iter: 3 and result: ({3, 4}, [3, 3])


## 3.2 Greedy algorithm with empty set

In [11]:
def greedy_algorithm_empty_set(
    G: nx.Graph, terminals: list[int], 
    k: int, case: 1 | 2) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage

  T : set = set(terminals)
  V : set = set(G.nodes)

  S : set = set()
  pc_deltas : list[int] = []

  K : set = set()

  if len(V) - len(T) == k:
    return V - T, [0]
  
  if case == 1:
    K = set()

  elif case == 2:
    K = T

  while len(K) != len(V) - k:

    best_j = None
    best_pc = float("inf")

    for j in V - K:
      
      S_j = V - (K | {j})
      pc_j = compute_pc(G, S_j, terminal_nodes=terminals)

      # print(pc_j)
      
      # argmin
      if pc_j < best_pc:
        best_pc = pc_j
        best_j = j

    K.add(best_j)
    pc_deltas.append(best_pc)

  S = V - K

  return S, pc_deltas

In [12]:
T = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(T)
G.nodes(data="terminal")

result = greedy_algorithm_empty_set(G, T, k=2, case=1)
result

({3, 5}, [0])

In [13]:
T = [1, 2, 4, 6, 7]
G = create_custom_graph_with_2_comps(T)
G.nodes(data="terminal")

result = greedy_algorithm_empty_set(G, T, k=2, case=2)
result

({3, 5}, [0])

In [14]:
T = [1, 2, 6]
G = create_custom_graph_extreme(T, edge_arrangement=3)
G.nodes(data="terminal")

result = greedy_algorithm_empty_set(G, T, k=2, case=1)
result

({2, 6}, [0, 0, 0, 0])

In [15]:
for i in range(3):
  T = [1, 2, 6]
  G = create_custom_graph_extreme(T, edge_arrangement=i + 1)
  G.nodes(data="terminal")

  result = greedy_algorithm_empty_set(G, T, k=2, case=2)
  print(f"iter: {i + 1} and result: {result}")

iter: 1 and result: ({3, 5}, [1])
iter: 2 and result: ({4, 5}, [1])
iter: 3 and result: ({4, 5}, [1])


# 4. Greedy with MIS

In [16]:
def greedy_mis_algorithm(
    G: nx.Graph, terminals: list[int], 
    k: int, case: int, mis_trails: int = 30 ) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage
  
  S : set = set()
  V : set = set(G.nodes)
  T : set = set(terminals)

  # Loop for selecting best warm-up MIS
  # which has resulted in lower total PC
  best_pc = float("inf")
  MIS : set = set()

  for _ in range(mis_trails):
    mis = set(nx.maximal_independent_set(G))

    # If case 2, ensure terminal nodes are not deleted
    # keep terminal nodes in MIS
    if case == 2:
      mis = mis | T
    
    # deletions correspond to V \ MIS
    S0 = V - mis

    # feasibility for case 2: do not delete terminal nodes
    if case == 2 and (S0 & T):
      continue

    curr_pc = compute_pc(G=G, S=S0, terminal_nodes=terminals)

    if curr_pc < best_pc:
      best_pc = curr_pc
      MIS = mis

  # print(f"mis: {MIS}")
  
  pc_deltas : list[int] = []

  while len(MIS) != len(V) - k:

    best_node = None
    best_deg = -1
    best_pc = float("inf")

    for node in V - MIS:

      if case == 1:
        # case 1: S in V
        S_j = V - (MIS | {node})
        pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)
        # print(f"node: {node} and pc Sj : {pc_Sj}")
        
      if case == 2:
        # case 2: S in V \ T
        if node not in T:
          S_j = V - (MIS | {node})
          pc_Sj = compute_pc(G=G, S=S_j, terminal_nodes=terminals)

      # tie-breaker - if two nodes give the same best_pc
      # get the one with higher degree
      deg = G.degree(node)

      # argmin
      if pc_Sj < best_pc or (pc_Sj == best_pc and deg > best_deg):
        best_pc = pc_Sj
        best_node = node
        best_deg = deg
        # print(f"best node: {best_node} and best gain: {best_pc}")
    
    if best_node is None:
      # No feasible solution
      break

    MIS.add(best_node)
    pc_deltas.append(best_pc)
  
    # print(f"mis iter: {MIS}")  
  
  S = V - MIS
  print(f"mis: {MIS} and S: {S}")

  return S, pc_deltas

In [17]:
for i in range(2):
  T = [1, 2, 4, 6, 7]
  G = create_custom_graph_with_2_comps(T)
  G.nodes(data="terminal")

  result = greedy_mis_algorithm(G, T, k=2, case=i + 1)
  
  print(f"iter: {i + 1} and result: {result}")


mis: {2, 3, 4, 5, 7} and S: {1, 6}
iter: 1 and result: ({1, 6}, [1])
mis: {1, 2, 4, 6, 7} and S: {3, 5}
iter: 2 and result: ({3, 5}, [])


In [18]:
for case in range(2):
  for i in range(3):
    T = [1, 2, 6]
    G = create_custom_graph_extreme(T, edge_arrangement=i + 1)
    G.nodes(data="terminal")

    result = greedy_mis_algorithm(G, T, k=2, case=case + 1)
    print("~~~~~~~")
    print(f"case: {case+1} - iter: {i + 1} and result: {result}")
    print("~~~~~~~")

mis: {2, 4, 5, 6} and S: {1, 3}
~~~~~~~
case: 1 - iter: 1 and result: ({1, 3}, [1])
~~~~~~~
mis: {1, 3, 4, 5} and S: {2, 6}
~~~~~~~
case: 1 - iter: 2 and result: ({2, 6}, [0])
~~~~~~~
mis: {2, 3, 4, 6} and S: {1, 5}
~~~~~~~
case: 1 - iter: 3 and result: ({1, 5}, [1])
~~~~~~~
mis: {1, 2, 4, 6} and S: {3, 5}
~~~~~~~
case: 2 - iter: 1 and result: ({3, 5}, [])
~~~~~~~
mis: {1, 2, 3, 6} and S: {4, 5}
~~~~~~~
case: 2 - iter: 2 and result: ({4, 5}, [])
~~~~~~~
mis: {1, 2, 3, 6} and S: {4, 5}
~~~~~~~
case: 2 - iter: 3 and result: ({4, 5}, [])
~~~~~~~


In [ ]:
def greedy_algorithm_mis_ashwin(
    G: nx.Graph, terminals: list[int], 
    k: int, case: 1 | 2) -> tuple[set, list[int]]:
  
  # If graph is too big, use sparse representation
  # for better memory usage

  T : set = set(terminals)
  V : set = set(G.nodes)

  S : set = set()
  pc_deltas : list[int] = []

  if case == 1:
    MIS = set(nx.maximal_independent_set(G))

    print(f"mis: {MIS}")
  
    while len(MIS) < len(V) - k:

      best_j = None
      best_pc = float("inf")

      for j in V - MIS:
        
        S_j = V - (MIS | {j})
        pc_j = compute_pc(G, S_j, terminal_nodes=terminals)

        # print(pc_j)
        
        # argmin
        if pc_j < best_pc:
          best_pc = pc_j
          best_j = j

      MIS.add(best_j)
      pc_deltas.append(best_pc)

    S = V - MIS

  elif case == 2:
    # H = G[V \ T]
    H = G.copy()
    H.remove_nodes_from(T)

    MIS = set(nx.maximal_independent_set(H))

    # MIS = MIS | T
    
    print(f"mis: {MIS}")
    
    # number of nonterminal nodes that can be kept
    r = len(V) - len(T) - k

    K = set()

    while len(K) != r:
      
      best_j = None
      best_pc = float("inf")

      for j in MIS:

        S_j = V - (K | {j})
        pc_j = compute_pc(G, S_j, terminal_nodes=terminals)

      if pc_j < best_pc:
        best_pc = pc_j
        best_j = j
    
      K.add(best_j)
      pc_deltas.append(best_pc)

    S = V - K

  print(f"kept final: {MIS}")
  
  return S, pc_deltas

In [76]:
def extended_critical_node_mis(G, terminals, k, case, maxIter):
  S = None

  best_S = None
  best_pc = float("inf")
  
  for _ in range(maxIter):

    result = greedy_algorithm_mis_ashwin(G, terminals=terminals, k=k, case=case)

    # print(result)

    S = result[0]
    print(S)
    curr_pc = compute_pc(G, S, terminal_nodes=terminals)

    print(curr_pc)
    
    if curr_pc < best_pc:
      best_pc = curr_pc
      best_S = S
    
  return best_S, best_pc

In [86]:
for i in range(2):
  T = [1, 2, 4, 6, 7]
  G = create_custom_graph_with_2_comps(T)
  G.nodes(data="terminal")

  result = extended_critical_node_mis(G, T, k=2, case=i + 1, maxIter=3)
  
  print(f"iter: {i + 1} and result: {result}")


mis start (case 1): {1, 3, 6}
kept final (case 1): {1, 3, 4, 5, 6}
{2, 7}
0
mis start (case 1): {2, 4, 5, 7}
kept final (case 1): {1, 2, 4, 5, 7}
{3, 6}
1
mis start (case 1): {2, 4, 5, 7}
kept final (case 1): {1, 2, 4, 5, 7}
{3, 6}
1
iter: 1 and result: ({2, 7}, 0)
start kept (case 2): {1, 2, 4, 6, 7} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 4, 6, 7}
{3, 5}
2
start kept (case 2): {1, 2, 4, 6, 7} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 4, 6, 7}
{3, 5}
2
start kept (case 2): {1, 2, 4, 6, 7} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 4, 6, 7}
{3, 5}
2
iter: 2 and result: ({3, 5}, 2)


In [88]:
for i in range(3):
  T = [1, 2, 6]
  G = create_custom_graph_extreme(T, edge_arrangement=i + 1)
  G.nodes(data="terminal")

  result = extended_critical_node_mis(G, T, k=2, case=1, maxIter=3)
  print("~~~~~~~")
  print(f"case: 1 - iter: {i + 1} and result: {result}")
  print("~~~~~~~")

mis start (case 1): {1, 3, 5}
kept final (case 1): {1, 3, 4, 5}
{2, 6}
0
mis start (case 1): {1, 3, 5}
kept final (case 1): {1, 3, 4, 5}
{2, 6}
0
mis start (case 1): {1, 3, 5}
kept final (case 1): {1, 3, 4, 5}
{2, 6}
0
~~~~~~~
case: 1 - iter: 1 and result: ({2, 6}, 0)
~~~~~~~
mis start (case 1): {2, 3, 6}
kept final (case 1): {1, 2, 3, 6}
{4, 5}
1
mis start (case 1): {1, 4, 5}
kept final (case 1): {1, 3, 4, 5}
{2, 6}
0
mis start (case 1): {2, 3, 6}
kept final (case 1): {1, 2, 3, 6}
{4, 5}
1
~~~~~~~
case: 1 - iter: 2 and result: ({2, 6}, 0)
~~~~~~~
mis start (case 1): {2, 3, 6}
kept final (case 1): {1, 2, 3, 6}
{4, 5}
1
mis start (case 1): {2, 3, 6}
kept final (case 1): {1, 2, 3, 6}
{4, 5}
1
mis start (case 1): {1, 4, 5}
kept final (case 1): {1, 3, 4, 5}
{2, 6}
0
~~~~~~~
case: 1 - iter: 3 and result: ({2, 6}, 0)
~~~~~~~


In [87]:
for i in range(3):
  T = [1, 2, 6]
  G = create_custom_graph_extreme(T, edge_arrangement=i + 1)
  G.nodes(data="terminal")

  result = extended_critical_node_mis(G, T, k=2, case=2, maxIter=3)
  print("~~~~~~~")
  print(f"case: 2 - iter: {i + 1} and result: {result}")
  print("~~~~~~~")

start kept (case 2): {1, 2, 3, 6} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 3, 6}
{4, 5}
3
start kept (case 2): {1, 2, 3, 6} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 3, 6}
{4, 5}
3
start kept (case 2): {1, 2, 3, 6} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 3, 6}
{4, 5}
3
~~~~~~~
case: 2 - iter: 1 and result: ({4, 5}, 3)
~~~~~~~
start kept (case 2): {1, 2, 3, 6} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 3, 6}
{4, 5}
1
start kept (case 2): {1, 2, 4, 6} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 4, 6}
{3, 5}
3
start kept (case 2): {1, 2, 4, 6} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 4, 6}
{3, 5}
3
~~~~~~~
case: 2 - iter: 2 and result: ({4, 5}, 1)
~~~~~~~
start kept (case 2): {1, 2, 3, 6} (terminals + MIS nonterm warm-start)
kept final (case 2): {1, 2, 3, 6}
{4, 5}
1
start kept (case 2): {1, 2, 3, 6} (terminals + MIS nonterm warm-start)
kept final (cas

In [85]:
def greedy_algorithm_mis_ashwin(
    G: nx.Graph, terminals: list[int], 
    k: int, case: 1 | 2) -> tuple[set, list[int]]:
  T = set(terminals)
  V = set(G.nodes)
  pc_deltas = []

  if case == 1:
    MIS = set(nx.maximal_independent_set(G))
    print(f"mis start (case 1): {MIS}")

    while len(MIS) < len(V) - k:
      best_j = None
      best_pc = float("inf")
      for j in V - MIS:
        S_j = V - (MIS | {j})
        pc_j = compute_pc(G, S_j, terminal_nodes=terminals)
        if pc_j < best_pc:
          best_pc = pc_j
          best_j = j
      if best_j is None:
        break
      MIS.add(best_j)
      pc_deltas.append(best_pc)

    S = V - MIS
    print(f"kept final (case 1): {MIS}")

  else:
    # MIS-based greedy keep selection with terminals fixed (case 2)
    H = G.copy()
    H.remove_nodes_from(T)
    MIS_nonterm = set(nx.maximal_independent_set(H))
    kept = set(T)

    # Warm-start by keeping MIS nonterm nodes first (if needed)
    for j in MIS_nonterm:
      if len(kept) >= len(V) - k:
        break
      if j not in kept:
        kept.add(j)

    print(f"start kept (case 2): {kept} (terminals + MIS nonterm warm-start)")

    while len(kept) < len(V) - k:
      best_j = None
      best_pc = float("inf")
      for j in V - kept:
        if j in T:
          continue
        S_j = V - (kept | {j})
        pc_j = compute_pc(G, S_j, terminal_nodes=terminals)
        if pc_j < best_pc:
          best_pc = pc_j
          best_j = j
      if best_j is None:
        break
      kept.add(best_j)
      pc_deltas.append(best_pc)

    S = V - kept
    print(f"kept final (case 2): {kept}")

  return S, pc_deltas